In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

data = np.array([10, 15, 18, 30, 12, 20, 21, 23, 19], dtype=float) 

#2D 
data = data.reshape(-1, 1)

#LSTM models learn better if data is between 0 and 1, so we use MinMaxScaler to scale all values.
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

# Prepare dataset: last 3 days to predict next day
X, y = [], []
seq_len = 3
for i in range(len(scaled_data) - seq_len):
    X.append(scaled_data[i:i + seq_len])
    y.append(scaled_data[i + seq_len])

X = np.array(X)
y = np.array(y)

# Reshape for LSTM (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))


# LSTM Model
#return_sequences=False: we only care about the last output of the LSTM to pass to the next layer (which is Dense).
#input_shape=(X.shape[1], 1): input shape is (sequence_length, number_of_features), which in your case is (3, 1).
#Dropout(0.2) Randomly sets 20% of inputs to zero during training to prevent overfitting.

model = Sequential([
    LSTM(50, return_sequences=False, input_shape=(X.shape[1], 1)),
    Dropout(0.2),
    Dense(1)
])

#X and y: Your training data. Each X is a sequence of 3 prices, and y is the next price.
#epochs=200: The model will go through the entire dataset 200 times, which helps it learn patterns better.
#verbose=0: No output is shown during training. You could change it to 1 or 2 if you want to see progress.

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=200, verbose=0)

# Predict next day (Day 10)
#This gets the last 3 entries from your scaled stock price data.
#Reshapes it into the required 3D input format for LSTM: (batch_size, sequence_length, number_of_features)
# In this case: 1 sample, 3 time steps, 1 feature per step.

last_3_days = scaled_data[-3:].reshape((1, seq_len, 1))

pred_scaled = model.predict(last_3_days)

#This scaled all stock prices into a range between 0 and 1, which helps the LSTM learn better.
#But after prediction, your output (pred_scaled) is also in that same 0–1 range.
#It takes that normalized value and converts it back to the real stock price.

pred = scaler.inverse_transform(pred_scaled)

print("📈 Predicted value for Day 10:", pred[0][0])


C:\Users\H.T\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step
📈 Predicted value for Day 10: 19.013063


In [11]:
    import numpy as np
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    
    # Sample labeled dataset (you can expand this)
    texts = [
        "I love this product",        # positive
        "This is the best ever",      # positive
        "Absolutely fantastic",       # positive
        "I hate this item",           # negative
        "Worst experience ever",      # negative
        "Very bad and terrible"       # negative
    ]
    
    labels = [1, 1, 1, 0, 0, 0]  # 1 = positive, 0 = negative
    
    # Tokenize
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    
    max_len = max(len(seq) for seq in sequences)
    vocab_size = len(tokenizer.word_index) + 1
    
    # Padding
    X = pad_sequences(sequences, maxlen=max_len)
    y = np.array(labels)
    
    # Model
    model = Sequential([
        Embedding(vocab_size, 10, input_length=max_len),
        LSTM(50),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X, y, epochs=200, verbose=0)
    
    # Prediction
    def predict_sentiment(text):
        seq = tokenizer.texts_to_sequences([text])
        padded = pad_sequences(seq, maxlen=max_len)
        pred = model.predict(padded, verbose=0)[0][0]
        return "Positive" if pred > 0.5 else "Negative"
    
    # Example
    print(predict_sentiment("Worst experience ever"))


Negative


In [5]:
import numpy as np
#Sequential model from Keras, which is used to build the neural network.
from tensorflow.keras.models import Sequential

#Embedding layer (used for representing words as vectors).v
from tensorflow.keras.layers import LSTM, Dense, Embedding

# Tokenize words, Tokenizer is used to convert text into sequences of integers.
from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.utils import to_categorical

#This is used to pad sequences to ensure they have the same length.
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Sample text
text = "AI is changing the world and transforming every industry rapidly"


#Creates a Tokenizer object, which is used to tokenize the text (i.e., split text into words and assign an index to each word).
tokenizer = Tokenizer()

#builds a word-to-index dictionary based on the words present in the text.
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

#This initializes a list to store the sequences of words.
# Create sequences: "AI is changing", then next word = "the"
input_sequences = []
words = text.split()
for i in range(3, len(words)):
    seq = words[i-3:i+1]  # 3 words + 1 next word
    token_list = tokenizer.texts_to_sequences([' '.join(seq)])[0] #The sequence ['AI', 'is', 'changing'] is converted into a numerical sequence using tokenizer.texts_to_sequences().
    input_sequences.append(token_list)
    
#numpy arry created
input_sequences = np.array(input_sequences) 

#1st part input x , last word output y
X, y = input_sequences[:, :-1], input_sequences[:, -1] 

#The target words (y) are converted to one-hot encoded vectors.
y = to_categorical(y, num_classes=total_words) 

# Model 
#An embedding layer that maps each word in the vocabulary to a vector of size 10.
#LSTM(50): An LSTM layer with 50 units to learn sequential dependencies.
model = Sequential([
    Embedding(total_words, 10, input_length=3),
    LSTM(50),
    Dense(total_words, activation='softmax')
])

#The model is compiled with categorical cross-entropy loss (for multi-class classification)

#verbose=0: No output. This makes the prediction silent, which is usually what you want inside functions or loops.
#verbose=1: Shows a progress bar during prediction (only really noticeable for large batches).
#verbose=2: Shows one line per sample (less common for prediction).

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=200, verbose=0)

# Predict next word
def predict_next(input_text):
    token_seq = tokenizer.texts_to_sequences([input_text])[0] #Converts the input text into a sequence of integers.
    token_seq = pad_sequences([token_seq], maxlen=3) #Pads the sequence to ensure it has a length of 3.
    pred_index = np.argmax(model.predict(token_seq, verbose=0))
    for word, index in tokenizer.word_index.items():
        if index == pred_index:
            return word

# Example
print("Next word after 'changing the world':", predict_next("changing the world"))


Next word after 'changing the world': and
